# Module 07 — Advanced Topics

This module covers production-grade snapshot patterns and edge cases:

| Section | Topic |
|---------|-------|
| 1 | Searchable snapshots (full_copy and shared_cache) |
| 2 | Snapshot cloning |
| 3 | Cross-cluster restore via read-only URL repo |
| 4 | Repository analysis (`_analyze` API) |
| 5 | Concurrent snapshot limits |
| 6 | Rate limiting (throttle bytes-per-sec) |
| 7 | Handling failures — partial snapshots & `_status` interpretation |
| 8 | Security feature state backup & restore |

In [ ]:
import time
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

# Ensure the baseline snapshot exists from Module 00
try:
    es.snapshot.get(repository=FS_REPO_NAME, snapshot='baseline')
    info('Baseline snapshot found.')
except Exception:
    warn('Baseline snapshot not found — run Module 00 first.')

---
## 1. Searchable Snapshots

Searchable snapshots let you **query data directly from a snapshot repository** without fully
restoring it. The shard data stays in the repository; Elasticsearch mounts a virtual index
that reads from it on demand.

Two mount strategies:

| Strategy | Storage impact | Use case |
|----------|---------------|----------|
| `full_copy` | Full local copy cached on node | Warm tier — faster queries, no network per-read |
| `shared_cache` | Partial LRU cache shared across mounted indices | Cold/frozen tier — minimal local storage |

In [ ]:
heading('1a. Mount searchable snapshot — full_copy strategy')

# Remove any previous mount
es.indices.delete(index='ecommerce-searchable-full', ignore_unavailable=True)

# Mount the ecommerce index from the baseline snapshot
es.searchable_snapshots.mount(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'index': 'kibana_sample_data_ecommerce',
        'renamed_index': 'ecommerce-searchable-full',
        'index_settings': {
            'index.number_of_replicas': 0,
        },
    },
    storage='full_copy',
    wait_for_completion=True,
)

# Wait for it to go green
for _ in range(30):
    health = es.cluster.health(index='ecommerce-searchable-full', wait_for_status='green', timeout='2s')
    if health['status'] == 'green':
        break
    time.sleep(1)

count = es.count(index='ecommerce-searchable-full')['count']
success(f'Mounted full_copy searchable snapshot: {count} documents')

# Inspect the special index settings
settings = es.indices.get_settings(index='ecommerce-searchable-full')
idx = settings['ecommerce-searchable-full']['settings']['index']
console.print(f'  store.type       : {idx.get("store", {}).get("type", "?")} ')
console.print(f'  tier preference  : {idx.get("routing", {}).get("allocation", {}).get("include", {}).get("_tier_preference", "?")} ')

In [ ]:
heading('1b. Query the mounted index normally')

# Standard search — transparent to the client
resp = es.search(
    index='ecommerce-searchable-full',
    body={
        'size': 3,
        'query': {'match_all': {}},
        '_source': ['customer_full_name', 'taxful_total_price', 'order_date'],
    },
)
info(f'Hits: {resp["hits"]["total"]["value"]}')
for hit in resp['hits']['hits']:
    s = hit['_source']
    console.print(f"  {s['customer_full_name']}  ${s['taxful_total_price']}  {s['order_date']}")

In [ ]:
heading('1c. Searchable snapshot stats')

try:
    stats = es.searchable_snapshots.stats(index='ecommerce-searchable-full')
    total = stats.get('total', {})
    console.print(f'  size_in_bytes       : {total.get("size_in_bytes", "?")} ')
    console.print(f'  hits (cache)        : {total.get("hits", "?")} ')
    console.print(f'  misses (disk reads) : {total.get("misses", "?")} ')
except Exception as e:
    warn(f'Stats unavailable: {e}')

# Unmount by deleting the index
es.indices.delete(index='ecommerce-searchable-full')
success('Unmounted (deleted) searchable snapshot index')

### Searchable snapshots — key points

- **Read-only**: you cannot index documents into a mounted index.
- **No replicas by default**: the repository is the source of truth; replicas would just
  duplicate the remote reads.
- **ILM integration**: the `searchable_snapshot` ILM action mounts warm/cold/frozen indices
  automatically when an index ages into those phases.
- **Frozen tier** uses `shared_cache` — a single cache file shared among all frozen shards
  on a node (`xpack.searchable.snapshot.shared_cache.size`).
- **Cannot snapshot a mounted index** — you must snapshot the original data tier instead.

---
## 2. Snapshot Cloning

Cloning copies a subset of indices from one snapshot into a **new snapshot in the same
repository** without transferring data over the network — only the repository metadata
is duplicated; shard files are referenced by both snapshots via deduplication.

**Use cases:**
- Pin a "known-good" copy of a snapshot before running destructive tests
- Split a large snapshot into smaller named snapshots per team / per index group
- Version a snapshot before an upgrade

In [ ]:
heading('2. Clone snapshot')

# Ensure the source snapshot exists
delete_snapshot_if_exists(es, FS_REPO_NAME, 'baseline-clone')

es.snapshot.clone(
    repository=FS_REPO_NAME,
    snapshot='baseline',        # source snapshot
    target_snapshot='baseline-clone',  # new snapshot name
    body={
        'indices': 'kibana_sample_data_ecommerce,kibana_sample_data_flights',
    },
)

# Wait for clone to complete
clone_snap = wait_for_snapshot(es, FS_REPO_NAME, 'baseline-clone')
success(f'Clone complete: {clone_snap["snapshot"]}  indices={clone_snap["indices"]}')

# Compare sizes — clones share shard files, so the clone adds almost no disk space
orig  = es.snapshot.get(repository=FS_REPO_NAME, snapshot='baseline')['snapshots'][0]
clone = es.snapshot.get(repository=FS_REPO_NAME, snapshot='baseline-clone')['snapshots'][0]

t = Table('Snapshot', 'Indices', 'Start time')
t.add_row('baseline',       str(len(orig['indices'])),  orig['start_time'])
t.add_row('baseline-clone', str(len(clone['indices'])), clone['start_time'])
console.print(t)

info('Note: both snapshots share shard segment files on disk — no data duplication.')

---
## 3. Cross-Cluster Restore via Read-Only URL Repository

The pattern for restoring to a **different cluster**:

```
Cluster A                Cluster B
─────────────────        ──────────────────────
fs-repo (source)  ──►   url-repo  ──►  _restore
(write snapshots)        (read-only view)
```

In this training environment both "clusters" are the same node, but the `url` repo type
gives a faithful read-only view of the filesystem repository — demonstrating that Cluster B
has no write access to the source repository.

In [ ]:
heading('3. Cross-cluster restore simulation')

# Register a read-only URL view of the fs repository (simulates Cluster B's repo)
try:
    es.snapshot.get_repository(name='url-cross-cluster')
    info('url-cross-cluster repo already registered.')
except Exception:
    es.snapshot.create_repository(
        name='url-cross-cluster',
        body={
            'type': 'url',
            'settings': {
                'url': f'file://{FS_REPO_PATH}',
            },
        },
    )
    success('Registered url-cross-cluster (read-only view of fs-repo)')

# Verify it is read-only — attempting to create a snapshot must fail
try:
    es.snapshot.create(
        repository='url-cross-cluster',
        snapshot='should-fail',
        body={},
        wait_for_completion=True,
    )
    warn('ERROR: snapshot creation should have been rejected by read-only repo!')
except Exception as e:
    success(f'Write correctly rejected: {type(e).__name__}')

# List available snapshots through the read-only repo
snaps = es.snapshot.get(repository='url-cross-cluster', snapshot='*')
info(f'Snapshots visible via url-cross-cluster: {[s["snapshot"] for s in snaps["snapshots"]]}')

# Restore from the read-only repo
es.indices.delete(index='ecommerce-cross-cluster', ignore_unavailable=True)
es.snapshot.restore(
    repository='url-cross-cluster',
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-cross-cluster',
        'include_global_state': False,
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'ecommerce-cross-cluster')
count = es.count(index='ecommerce-cross-cluster')['count']
success(f'Cross-cluster restore succeeded: {count} documents in ecommerce-cross-cluster')
es.indices.delete(index='ecommerce-cross-cluster')

---
## 4. Repository Analysis

`POST /_snapshot/{repo}/_analyze` performs a **thorough read/write/delete test** against
the repository to detect consistency, latency, and throughput issues.

Key parameters:

| Parameter | Default | Meaning |
|-----------|---------|--------|
| `blob_count` | 100 | Number of test blobs to write |
| `concurrency` | 10 | Parallel read/write threads |
| `read_node_count` | 10 | Nodes that read each blob |
| `early_read_node_count` | 2 | Nodes that read before write completes (tests consistency) |
| `seed` | random | RNG seed for reproducibility |
| `timeout` | 120s | Max analysis time |

In [ ]:
heading('4. Repository analysis')

info('Running _analyze — this may take ~30 seconds on a local fs repo...')
try:
    analysis = es.snapshot.repository_analyze(
        name=FS_REPO_NAME,
        blob_count=10,      # reduced for speed in training
        concurrency=2,
        read_node_count=1,
        early_read_node_count=0,
        timeout='60s',
    )

    summary = analysis.get('summary', {})
    listing = analysis.get('listing_elapsed', {})
    delete  = analysis.get('delete_elapsed', {})

    t = Table('Metric', 'Value')
    t.add_row('Blob count',            str(analysis.get('blob_count', '?')))
    t.add_row('Concurrency',           str(analysis.get('concurrency', '?')))
    t.add_row('Write success',         str(summary.get('write', {}).get('count', '?')))
    t.add_row('Read success',          str(summary.get('read', {}).get('count', '?')))
    t.add_row('Listing elapsed (ms)',  str(listing.get('millis', '?')))
    t.add_row('Delete elapsed (ms)',   str(delete.get('millis', '?')))
    console.print(t)

    success('Repository analysis passed — no consistency issues detected.')
except Exception as e:
    warn(f'Analysis error: {e}')

In [ ]:
heading('4b. Repository verify and cleanup')

# Verify — confirms every node can write to and read from the repository
verify = es.snapshot.verify_repository(name=FS_REPO_NAME)
nodes = verify.get('nodes', {})
info(f'Verify: nodes that confirmed access — {list(nodes.keys())}')

# Cleanup — removes orphaned snapshot files (e.g. from failed partial snapshots)
cleanup = es.snapshot.cleanup_repository(name=FS_REPO_NAME)
result = cleanup.get('results', {})
info(f'Cleanup: deleted_bytes={result.get("deleted_bytes", 0)}  deleted_blobs={result.get("deleted_blobs", 0)}')

---
## 5. Concurrent Snapshot Limits

Elasticsearch limits concurrent in-progress snapshots to protect cluster stability.

| Setting | Scope | Default |
|---------|-------|---------|
| `snapshot.max_concurrent_operations` | cluster | `1000` (ES 8+) |

Each in-progress snapshot counts as one operation. If you try to start more snapshots
than the limit allows, the API returns a `429 Too Many Requests` error.

In [ ]:
heading('5. Concurrent snapshot limits')

# Read the current setting
settings = es.cluster.get_settings(include_defaults=True)
defaults  = settings.get('defaults', {})
transient = settings.get('transient', {})
persistent = settings.get('persistent', {})

current = (
    transient.get('snapshot', {}).get('max_concurrent_operations') or
    persistent.get('snapshot', {}).get('max_concurrent_operations') or
    defaults.get('snapshot', {}).get('max_concurrent_operations') or
    'not set (default 1000)'
)
info(f'snapshot.max_concurrent_operations = {current}')

# Set a very low limit to demonstrate the guard
es.cluster.put_settings(body={'transient': {'snapshot.max_concurrent_operations': 1}})
info('Set max_concurrent_operations = 1')

# Start a legitimate snapshot (async)
delete_snapshot_if_exists(es, FS_REPO_NAME, 'concurrent-test-1')
delete_snapshot_if_exists(es, FS_REPO_NAME, 'concurrent-test-2')

es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='concurrent-test-1',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=False,
)
info('Started concurrent-test-1 (async)')

# Immediately try a second — should be rejected
try:
    es.snapshot.create(
        repository=FS_REPO_NAME,
        snapshot='concurrent-test-2',
        body={'indices': ['kibana_sample_data_flights'], 'include_global_state': False},
        wait_for_completion=False,
    )
    warn('Second snapshot started — limit may not have been enforced (snapshots can complete very quickly).')
except Exception as e:
    success(f'Second snapshot correctly rejected: {type(e).__name__}')

# Wait for first snapshot to finish, then restore the default
wait_for_snapshot(es, FS_REPO_NAME, 'concurrent-test-1')
es.cluster.put_settings(body={'transient': {'snapshot.max_concurrent_operations': None}})
success('Restored default max_concurrent_operations')

---
## 6. Rate Limiting

Two repository-level settings throttle I/O to prevent snapshots from saturating network
or disk bandwidth:

| Setting | Direction | Default |
|---------|-----------|--------|
| `max_snapshot_bytes_per_sec` | shard → repository | `40mb` |
| `max_restore_bytes_per_sec`  | repository → shard | unlimited |

These are set **per repository** (not per cluster) so you can throttle S3 repos differently
from local fs repos.

In [ ]:
heading('6. Rate-limit a repository')

# Update the fs repo with rate limits
es.snapshot.create_repository(
    name=FS_REPO_NAME,
    body={
        'type': 'fs',
        'settings': {
            'location': FS_REPO_PATH,
            'max_snapshot_bytes_per_sec': '10mb',
            'max_restore_bytes_per_sec':  '20mb',
        },
    },
)
success('Repository updated with rate limits: snapshot=10mb/s, restore=20mb/s')

# Verify by reading back
repo = es.snapshot.get_repository(name=FS_REPO_NAME)
settings = repo[FS_REPO_NAME]['settings']
console.print(f'  max_snapshot_bytes_per_sec : {settings.get("max_snapshot_bytes_per_sec", "(default)")} ')
console.print(f'  max_restore_bytes_per_sec  : {settings.get("max_restore_bytes_per_sec",  "(default)")} ')

# Remove rate limits (restore defaults)
es.snapshot.create_repository(
    name=FS_REPO_NAME,
    body={'type': 'fs', 'settings': {'location': FS_REPO_PATH}},
)
info('Rate limits removed — repository back to defaults.')

---
## 7. Handling Failures — Partial Snapshots & `_status`

### Snapshot states

| State | Meaning |
|-------|---------|
| `IN_PROGRESS` | Currently running |
| `SUCCESS` | Completed normally |
| `PARTIAL` | Completed but some shards failed |
| `FAILED` | All shards failed or fatal error |

A `PARTIAL` snapshot **is still restorable** for the shards that succeeded.
The `partial: true` restore option is required to use it.

### Shard-level failure information

Use `GET /_snapshot/{repo}/{snap}` and look at `shards_stats` and `indices[*].shards`
to find which shards failed and why.

In [ ]:
heading('7a. Inspect _status for an in-progress snapshot')

# Start a snapshot without waiting so we can observe _status
delete_snapshot_if_exists(es, FS_REPO_NAME, 'status-demo')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='status-demo',
    body={
        'indices': ['kibana_sample_data_*'],
        'include_global_state': False,
    },
    wait_for_completion=False,
)

# Poll _status a few times
seen_in_progress = False
for _ in range(20):
    try:
        status = es.snapshot.status(repository=FS_REPO_NAME, snapshot='status-demo')
        snap_status = status['snapshots'][0] if status['snapshots'] else None
        if snap_status:
            state = snap_status['state']
            stats = snap_status.get('stats', {})
            total_files   = stats.get('total', {}).get('file_count', 0)
            done_files    = stats.get('done', {}).get('file_count', 0) if 'done' in stats else stats.get('processed', {}).get('file_count', 0)
            total_bytes   = stats.get('total', {}).get('size_in_bytes', 0)
            console.print(f'  state={state}  files={done_files}/{total_files}  bytes={total_bytes}')
            if state == 'IN_PROGRESS':
                seen_in_progress = True
            if state in ('SUCCESS', 'PARTIAL', 'FAILED'):
                break
    except Exception:
        pass
    time.sleep(0.5)

if not seen_in_progress:
    info('Snapshot completed before we could catch IN_PROGRESS — local fs is very fast.')

final = wait_for_snapshot(es, FS_REPO_NAME, 'status-demo')
success(f'Final snapshot state: {final["state"]}')

In [ ]:
heading('7b. Detailed snapshot inspection — shards_stats')

snap_detail = es.snapshot.get(
    repository=FS_REPO_NAME,
    snapshot='status-demo',
)['snapshots'][0]

shards_stats = snap_detail.get('shards_stats', snap_detail.get('stats', {}))
console.print(f'  Snapshot     : {snap_detail["snapshot"]}')
console.print(f'  State        : {snap_detail["state"]}')
console.print(f'  Indices      : {snap_detail["indices"]}')
console.print(f'  Shards total : {snap_detail["shards"]["total"]}')
console.print(f'  Shards ok    : {snap_detail["shards"]["successful"]}')
console.print(f'  Shards failed: {snap_detail["shards"]["failed"]}')

if snap_detail['shards']['failed'] > 0:
    warn('Some shards failed — inspect failures below:')
    for failure in snap_detail.get('failures', []):
        warn(f'  index={failure["index"]}  shard={failure["shard"]}  reason={failure.get("reason", "?")}')
else:
    success('All shards succeeded.')

In [ ]:
heading('7c. partial: true — restore a PARTIAL or SUCCESS snapshot allowing missing shards')

# In normal circumstances all shards succeed; this demonstrates the API parameter
# used when restoring a PARTIAL snapshot (some shards missing).
es.indices.delete(index='ecommerce-partial-restore', ignore_unavailable=True)

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='status-demo',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-partial-restore',
        'include_global_state': False,
        'partial': True,   # allow restore even if some shards are missing from the snapshot
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'ecommerce-partial-restore')
count = es.count(index='ecommerce-partial-restore')['count']
success(f'Partial-allowed restore succeeded: {count} documents')

es.indices.delete(index='ecommerce-partial-restore')

---
## 8. Security Feature State — Backup & Restore

The `security` feature state captures:

| Object type | Notes |
|-------------|-------|
| Native users | Passwords stored as hashed values |
| Roles | Custom role definitions |
| Role mappings | Realm-to-role assignments |
| API keys | Hashed, restored but **not** usable — must reissue |
| Service tokens | Same caveat as API keys |
| Application privileges | Custom app-level privileges |

> **Important:** Restoring the `security` feature state **overwrites** the target cluster's
> security objects. Always restore to a clean or isolated cluster when testing.
> Built-in users (`elastic`, `kibana_system`, etc.) and their passwords are **not** overwritten.

In [ ]:
heading('8a. Create a test user and role to snapshot')

# Create a custom role
es.security.put_role(
    name='training-reader',
    body={
        'cluster': ['monitor'],
        'indices': [{
            'names': ['kibana_sample_data_*'],
            'privileges': ['read', 'view_index_metadata'],
        }],
    },
)
success('Role created: training-reader')

# Create a user assigned to that role
es.security.put_user(
    username='training-user',
    body={
        'password': 'TrainingUser123!',
        'roles': ['training-reader'],
        'full_name': 'Training Demo User',
        'email': 'training@example.com',
    },
)
success('User created: training-user  (role: training-reader)')

# Verify
users = es.security.get_user(username='training-user')
console.print(f'  Roles: {users["training-user"]["roles"]}')

In [ ]:
heading('8b. Snapshot only the security feature state')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'security-backup')

es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='security-backup',
    body={
        'indices': [],                    # no data indices
        'include_global_state': False,    # no cluster state
        'feature_states': ['security'],   # ONLY security objects
    },
    wait_for_completion=True,
)

snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='security-backup')['snapshots'][0]
success(f'Security snapshot created: {snap["snapshot"]}  state={snap["state"]}')
console.print(f'  Feature states: {[fs["feature_name"] for fs in snap.get("feature_states", [])]}')

In [ ]:
heading('8c. Delete user & role, then restore from snapshot')

# Delete them to simulate loss
es.security.delete_user(username='training-user')
es.security.delete_role(name='training-reader')
warn('Deleted training-user and training-reader role')

# Confirm they are gone
try:
    es.security.get_user(username='training-user')
    warn('User still exists!')
except Exception:
    info('Confirmed: training-user does not exist.')

# Restore the security feature state
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='security-backup',
    body={
        'indices': [],
        'include_global_state': False,
        'feature_states': ['security'],
    },
    wait_for_completion=True,
)

# Verify they are back
restored_user = es.security.get_user(username='training-user')
restored_role = es.security.get_role(name='training-reader')
success(f'User restored: {restored_user["training-user"]["full_name"]}')
success(f'Role restored: training-reader  cluster={restored_role["training-reader"]["cluster"]}')

info('')
info('Note: the user password hash IS restored, so the user can log in.')
info('API keys are NOT usable after restore — they must be reissued.')

In [ ]:
heading('8d. Clean up security objects')

es.security.delete_user(username='training-user')
es.security.delete_role(name='training-reader')
success('Cleaned up training-user and training-reader')

---
## Summary

| Topic | Key API / Setting |
|-------|-------------------|
| Searchable snapshots | `POST /_snapshot/{repo}/{snap}/_mount` with `storage=full_copy\|shared_cache` |
| Snapshot cloning | `PUT /_snapshot/{repo}/{snap}/_clone/{target}` |
| Cross-cluster restore | Register source as `url` repo on target cluster, then `_restore` |
| Repository analysis | `POST /_snapshot/{repo}/_analyze` |
| Repository verify | `POST /_snapshot/{repo}/_verify` |
| Repository cleanup | `POST /_snapshot/{repo}/_cleanup` |
| Concurrent limits | `snapshot.max_concurrent_operations` (cluster setting) |
| Rate limiting | `max_snapshot_bytes_per_sec` / `max_restore_bytes_per_sec` (repo setting) |
| Snapshot status | `GET /_snapshot/{repo}/{snap}/_status` |
| Partial restore | `partial: true` in restore body |
| Security backup | `feature_states: ["security"]` in snapshot body |

---

**Congratulations — you have completed all modules of the Elasticsearch Snapshot Training!**

Return to [00_setup.ipynb](00_setup.ipynb) to reset the environment and repeat any module.